# Lead-wise Transformer: Cross-lead ECG Classification

**Novel contribution:** After per-lead temporal encoding (shared weights), a cross-lead attention layer lets each lead attend to all other leads — mirroring how cardiologists compare leads in diagnosis (e.g. ST elevation in leads II, III, aVF simultaneously for inferior MI). All other models in this project are channel-independent; this is the first to model inter-lead relationships explicitly.

In [ ]:
# Cell 1: Setup
import sys, os, warnings
sys.path.append('../')
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import json
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt

from src.utils.config                import CFG
from src.preprocessing.label_utils   import load_all_labels
from src.preprocessing.dataset_full  import ECGDatasetFull
from src.models.leadwise_transformer import LeadwiseTransformer
from src.training.train_peft         import run_experiment

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB")

In [ ]:
# Cell 2: Load data
DATA_PATH = CFG['data']['path']
Y = load_all_labels(DATA_PATH + 'ptbxl_database.csv', DATA_PATH + 'scp_statements.csv')
train_df = Y[Y.strat_fold <  9]
val_df   = Y[Y.strat_fold == 9]
test_df  = Y[Y.strat_fold == 10]

train_ds = ECGDatasetFull(train_df, DATA_PATH)
val_ds   = ECGDatasetFull(val_df,   DATA_PATH)

print(f"Train: {len(train_df)} records  ({len(train_ds)} samples)")
print(f"Val:   {len(val_df)} records  ({len(val_ds)} samples)")
print(f"Test:  {len(test_df)} records (held out)")

In [ ]:
# Cell 3: Architecture inspection
model = LeadwiseTransformer()
params = model.count_parameters()
print(f"\nParameter counts:")
print(f"  Trainable: {params['trainable']:,}")
print(f"  Total:     {params['total']:,}")
print(f"  Ratio:     {params['percentage']}")

dummy = torch.randn(4, 12, 1000)
with torch.no_grad():
    out = model(dummy)
assert out.shape == (4, 5), f"Expected (4, 5), got {out.shape}"
print(f"\nForward pass OK: {dummy.shape} -> {out.shape}")

print("\nNamed modules:")
for name, mod in model.named_children():
    print(f"  {name}: {mod.__class__.__name__}")

In [ ]:
# Cell 4: Train lead-wise transformer
auc, hist = run_experiment(
    model, train_ds, val_ds,
    experiment_name = 'leadwise_transformer',
    epochs          = CFG['training']['epochs'],
    lr              = CFG['training']['lr_peft'],
    batch_size      = CFG['training']['batch_size_full'],
    save_dir        = CFG['paths']['results'],
)
del model; torch.cuda.empty_cache()
print(f"Best AUC: {auc:.4f}")

In [ ]:
# Cell 5: Load all experiment results
RESULTS      = CFG['paths']['results']
SUPERCLASSES = CFG['data']['superclasses']

def load_hist(name):
    path = os.path.join(RESULTS, name, "history.json")
    if not os.path.exists(path):
        print(f"  Skipping {name} (no history.json)")
        return None
    with open(path) as f:
        return json.load(f)

hist_hubert4  = load_hist('hubert_ecg_blocks4')
hist_hubert8  = load_hist('hubert_ecg_blocks8')
hist_lora     = load_hist('hubert_ecg_lora_r8')
hist_dora     = load_hist('hubert_ecg_dora_r8')
hist_leadwise = load_hist('leadwise_transformer')

with open(os.path.join(RESULTS, 'dummy_metrics.json')) as f:
    dummy_m = json.load(f)
with open(os.path.join(RESULTS, 'baseline_cnn', 'baseline_cnn_metrics.json')) as f:
    cnn_m = json.load(f)

print("Loaded:")
for name, h in [('HuBERT 4 blocks',       hist_hubert4),
                ('HuBERT 8 blocks',       hist_hubert8),
                ('HuBERT LoRA r=8',       hist_lora),
                ('HuBERT DoRA r=8',       hist_dora),
                ('Lead-wise Transformer', hist_leadwise)]:
    if h:
        n_ep = len(h["history"])
        print(f"  {name}: best AUC={h['best_auc']:.4f}  ({n_ep} epochs)")

In [ ]:
# Cell 6: Full comparison results table
def best_auc(h): return h['best_auc'] if h else float('nan')
def best_f1(h):
    if not h: return float('nan')
    return max(e["f1_macro"] for e in h["history"])

cnn_auc = cnn_m['auc_macro']
rows = [
    {'Model': 'Dummy (prior)',      'Strategy': 'Prior freq.',       'AUC': dummy_m['auc_macro'], 'F1': dummy_m['f1_macro']},
    {'Model': 'CNN (baseline)',     'Strategy': 'Full training',     'AUC': cnn_auc,              'F1': cnn_m['f1_macro']},
]
if hist_hubert4:
    rows.append({'Model': 'HuBERT-ECG-base',   'Strategy': 'Unfreeze 4 blocks', 'AUC': best_auc(hist_hubert4),  'F1': best_f1(hist_hubert4)})
if hist_hubert8:
    rows.append({'Model': 'HuBERT-ECG-base',   'Strategy': 'Unfreeze 8 blocks', 'AUC': best_auc(hist_hubert8),  'F1': best_f1(hist_hubert8)})
if hist_lora:
    rows.append({'Model': 'HuBERT-ECG-base',   'Strategy': 'LoRA r=8',          'AUC': best_auc(hist_lora),     'F1': best_f1(hist_lora)})
if hist_dora:
    rows.append({'Model': 'HuBERT-ECG-base',   'Strategy': 'DoRA r=8',          'AUC': best_auc(hist_dora),     'F1': best_f1(hist_dora)})
if hist_leadwise:
    rows.append({'Model': 'Lead-wise Transf.', 'Strategy': 'From scratch',      'AUC': best_auc(hist_leadwise), 'F1': best_f1(hist_leadwise)})

results = pd.DataFrame(rows)
results['vs CNN'] = (results['AUC'] - cnn_auc).map(
    lambda x: f'+{x:.4f}' if x > 0 else f'{x:.4f}'
)
print(results.to_string(index=False))

In [ ]:
# Cell 7: Learning curves
fig, (ax_auc, ax_loss) = plt.subplots(1, 2, figsize=(14, 5))

experiments = []
if hist_hubert4:  experiments.append(("HuBERT 4 blocks",       hist_hubert4["history"],  "tab:blue"))
if hist_hubert8:  experiments.append(("HuBERT 8 blocks",       hist_hubert8["history"],  "tab:orange"))
if hist_lora:     experiments.append(("HuBERT LoRA r=8",       hist_lora["history"],     "tab:purple"))
if hist_dora:     experiments.append(("HuBERT DoRA r=8",       hist_dora["history"],     "tab:brown"))
if hist_leadwise: experiments.append(("Lead-wise Transformer", hist_leadwise["history"], "tab:green"))

for name, h, color in experiments:
    ep = [e["epoch"] for e in h]
    ax_auc.plot(ep,  [e["auc_macro"] for e in h], label=name, color=color, linewidth=2)
    ax_loss.plot(ep, [e["val_loss"]  for e in h], label=name, color=color, linewidth=2)

ax_auc.axhline(y=cnn_m['auc_macro'],   color='red',  linestyle='--', linewidth=1.5, label='CNN baseline')
ax_auc.axhline(y=dummy_m['auc_macro'], color='gray', linestyle=':',  linewidth=1.5, label='Dummy baseline')
ax_auc.set_title('Validation AUC over epochs')
ax_auc.set_xlabel('Epoch'); ax_auc.set_ylabel('AUC (macro)')
ax_auc.legend(); ax_auc.grid(True, alpha=0.3)

ax_loss.set_title('Validation loss over epochs')
ax_loss.set_xlabel('Epoch'); ax_loss.set_ylabel('Loss')
ax_loss.legend(); ax_loss.grid(True, alpha=0.3)

plt.tight_layout()
os.makedirs(CFG['paths']['figures'], exist_ok=True)
plt.savefig(CFG['paths']['figures'] + 'all_experiments_comparison.png', dpi=150)
plt.show()

In [ ]:
# Cell 8: Per-class AUC grouped bar chart
fig, ax = plt.subplots(figsize=(13, 5))

plot_exps = []
for label, h in [("HuBERT 4 blocks",   hist_hubert4),
                 ("HuBERT 8 blocks",   hist_hubert8),
                 ("HuBERT LoRA r=8",   hist_lora),
                 ("HuBERT DoRA r=8",   hist_dora),
                 ("Lead-wise Transf.", hist_leadwise)]:
    if h:
        best_ep = max(h["history"], key=lambda e: e["auc_macro"])
        plot_exps.append((label, best_ep["per_class"]))

n_exp = len(plot_exps)
n_cls = len(SUPERCLASSES)
x     = np.arange(n_cls)
width = 0.7 / n_exp

for i, (name, per_class) in enumerate(plot_exps):
    vals   = [per_class.get(cls, 0) for cls in SUPERCLASSES]
    offset = (i - (n_exp - 1) / 2) * width
    ax.bar(x + offset, vals, width, label=name)

ax.axhline(y=0.5, color='gray', linestyle=':', linewidth=1, label='Chance level')
ax.set_xticks(x); ax.set_xticklabels(SUPERCLASSES)
ax.set_ylim(0, 1.05)
ax.set_title('Per-class AUC at best epoch')
ax.set_xlabel('Superclass'); ax.set_ylabel('AUC')
ax.legend(); ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig(CFG['paths']['figures'] + 'per_class_auc_comparison.png', dpi=150)
plt.show()

In [ ]:
# Cell 9: Best model analysis and full parameter efficiency table
all_res = []
for name, h in [("HuBERT-ECG-base (4 blocks)", hist_hubert4),
                ("HuBERT-ECG-base (8 blocks)", hist_hubert8),
                ("HuBERT-ECG-base (LoRA r=8)", hist_lora),
                ("HuBERT-ECG-base (DoRA r=8)", hist_dora),
                ("Lead-wise Transformer",       hist_leadwise)]:
    if h:
        best_ep = max(h["history"], key=lambda e: e["auc_macro"])
        all_res.append((name, h["best_auc"], best_ep["per_class"]))

if all_res:
    best_name, best_auc_val, best_pc = max(all_res, key=lambda r: r[1])
    print(f"Best model: {best_name}  (AUC={best_auc_val:.4f})")
    print("Per-class AUC breakdown:")
    for cls in SUPERCLASSES:
        v = best_pc.get(cls, 0)
        print(f"  {cls:5s}: {v:.4f}  {'#' * int(v * 20)}")



In [ ]:
# Parameter efficiency table 
from src.models.hubert_ecg_finetune import HuBERTECGClassifier, HuBERTECGPEFT

TOTAL = 93_323_397   # HuBERT-ECG-base confirmed total params

print(f"\nParameter Efficiency Comparison")
print(f"  {'Method':<28s}  {'Trainable':>14s}  {'% of total':>10s}")
print(f"  {'-'*58}")
print(f"  {'Full fine-tune':<28s}  {TOTAL:>14,}  {100.0:>9.1f}%")
print(f"  {'CNN (baseline)':<28s}  {'~50K':>14s}  {'<0.1%':>10s}")

m4 = HuBERTECGClassifier(size='base', blocks_to_unfreeze=4)
p4 = m4.count_parameters()
print(f"  {'Selective (4 blocks)':<28s}  {p4['trainable']:>14,}  {100*p4['trainable']/TOTAL:>9.1f}%")
del m4

m8 = HuBERTECGClassifier(size='base', blocks_to_unfreeze=8)
p8 = m8.count_parameters()
print(f"  {'Selective (8 blocks)':<28s}  {p8['trainable']:>14,}  {100*p8['trainable']/TOTAL:>9.1f}%")
del m8

ml = HuBERTECGPEFT(rank=8, use_dora=False)
pl = ml.count_parameters()
print(f"  {'LoRA r=8':<28s}  {pl['trainable']:>14,}  {100*pl['trainable']/pl['total']:>9.1f}%")
del ml; torch.cuda.empty_cache()

md = HuBERTECGPEFT(rank=8, use_dora=True)
pd_ = md.count_parameters()
print(f"  {'DoRA r=8':<28s}  {pd_['trainable']:>14,}  {100*pd_['trainable']/pd_['total']:>9.1f}%")
del md; torch.cuda.empty_cache()

lw = LeadwiseTransformer()
pl2 = lw.count_parameters()
print(f"  {'Lead-wise Transformer':<28s}  {pl2['trainable']:>14,}  {'100% (scratch)':>10s}")
del lw